# Module 3: Skills + Steering (15 min)

Add workflow knowledge via **skills** (markdown documents the agent can activate) and enforce business rules with **steering handlers** — a deterministic refund workflow enforcer and an LLM-based tone guardrail.

**Prerequisites:** Modules 1-2 completed

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# AWS-sponsored events / AWS credits
# If you are running this workshop with AWS-provided credits, those credits
# only work with Amazon Nova models — not Claude (the default).
# To switch, import BedrockModel and pass it to Agent(...):
#
# from strands.models import BedrockModel
# model = BedrockModel(model_id="amazon.nova-pro-v1:0")
# agent = Agent(model=model, tools=[...], plugins=[...], system_prompt=...)
#
# Available Nova model IDs: https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards-amazon.html
#   amazon.nova-micro-v1:0  — fastest, text-only, lowest cost
#   amazon.nova-lite-v1:0   — low-cost, multimodal (text, image, video)
#   amazon.nova-pro-v1:0    — balanced accuracy/speed, multimodal (recommended)
#
# Run locally without AWS credentials using Ollama (https://ollama.com/download):
#   1. Install Ollama — macOS: DMG at ollama.com | Linux: curl -fsSL https://ollama.com/install.sh | sh
#   2. ollama pull llama3.1   (supports tool use)
#   3. pip install strands-agents[ollama]
#   4. from strands.models import OllamaModel
#      model = OllamaModel(host="http://localhost:11434", model_id="llama3.1")
#      agent = Agent(model=model, tools=[...], plugins=[...], system_prompt=...)
#   Other models with tool support: llama3.2, qwen2.5, qwen3, mistral

---

## Part 1: Skills — Workflow Knowledge

Skills are markdown files (`SKILL.md`) that give the agent step-by-step procedures. The agent loads them on demand via the `AgentSkills` plugin.

Check out the `skills/` folder — it has three skills:
- `refund-processing` — how to handle refund requests
- `order-tracking` — how to check order status
- `account-troubleshooting` — how to handle account issues

In [ ]:
from strands import Agent, AgentSkills
from customer_service_tools import lookup_customer, get_order_history, process_refund

SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise. Use the available tools to look up customer
information and process requests. When a customer needs help, activate the appropriate
skill for step-by-step guidance.

Important guidelines:
- Always ask for the customer ID first if you don't have it.
- Use the data returned by tools to answer questions.
- Be warm but efficient."""

skills_plugin = AgentSkills(skills=["./skills"])

agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[skills_plugin],
    system_prompt=SYSTEM_PROMPT,
)

# The agent will activate the refund-processing skill to guide its workflow
result = agent("Hi, I'm customer C-1001. I'd like a refund for my wireless headphones.")

---

## Part 2: Steering Handlers — Deterministic Enforcement

Skills are suggestions. Steering handlers are **enforcement**. The `RefundWorkflowHandler` blocks `process_refund` unless the agent has already called `lookup_customer` and `get_order_history`.

In [ ]:
import re
from strands.vended_plugins.steering import (
    SteeringHandler, LLMSteeringHandler,
    Proceed, Guide, ToolSteeringAction,
    LedgerProvider,
)


class RefundWorkflowHandler(SteeringHandler):
    """Deterministic handler: enforce refund workflow ordering.

    Rules:
    1. Must look up the customer before processing a refund
    2. Must check order history before processing a refund
    """

    name = "refund-workflow"

    def __init__(self):
        super().__init__(context_providers=[LedgerProvider()])

    async def steer_before_tool(self, *, agent, tool_use, **kwargs) -> ToolSteeringAction:
        if tool_use.get("name") != "process_refund":
            return Proceed(reason="Not a refund operation")

        print("[STEERING] 🔍 process_refund attempted — checking workflow...")

        # Get execution history through managed ledger
        ledger = self.steering_context.data.get("ledger") or {}
        tool_calls = ledger.get("tool_calls", [])

        # Must look up customer first
        customer_verified = any(
            c["tool_name"] == "lookup_customer" and c["status"] == "success"
            for c in tool_calls
        )
        if not customer_verified:
            print("[STEERING] ⚠️  Blocked: must look up customer first")
            return Guide(
                reason="You must look up the customer with lookup_customer "
                "before processing a refund."
            )

        # Must check order history
        order_checked = any(
            c["tool_name"] == "get_order_history" and c["status"] == "success"
            for c in tool_calls
        )
        if not order_checked:
            print("[STEERING] ⚠️  Blocked: must check order history first")
            return Guide(
                reason="You must check the order history with get_order_history "
                "before processing a refund."
            )

        print("[STEERING] ✅ Refund workflow validated — proceeding")
        return Proceed(reason="Refund workflow validated")


print("✅ RefundWorkflowHandler defined")

---

## Part 3: LLM-Based Steering — Tone Guardrail

The `ToneGuardrailHandler` uses a second LLM call to evaluate the agent's response quality after each model turn.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field
from strands import Agent


class ToneVerdict(BaseModel):
    """Structured answer from the tone judge."""

    decision: Literal["proceed", "guide"] = Field(description="'proceed' if the reply follows the guidelines, 'guide' if it needs a fix")
    reason: str = Field(description="Short explanation, or what to fix")


class ToneGuardrailHandler(LLMSteeringHandler):
    """LLM-based handler: a second model call grades the TONE of the agent's final reply.

    Fix (AWS SBG at UH edition): LLMSteeringHandler only judges tool calls by default, so the
    original handler sent every lookup/refund call to a tone judge, which often cancelled them
    and made the agent retry tool calls over and over. Now tool calls always proceed and only
    the final text reply is graded, with at most one rewrite per reply.
    """

    name = "tone-guardrail"

    def __init__(self):
        super().__init__(
            system_prompt="""You are evaluating a customer service agent's reply to a customer.
Ensure the reply follows these communication guidelines:

- Don't overpromise or guarantee specific timelines beyond what the system confirms
- Don't blame the customer, other departments, or third parties
- Acknowledge the customer's frustration before jumping to solutions
- Don't offer discounts, credits, or compensation beyond what's in the workflow instructions
- Keep responses concise and actionable, no walls of text
- Never share internal system details, error codes, or technical jargon with the customer
- If the agent can't help, it should clearly explain why and offer an alternative

If the reply violates any of these, decide 'guide' and say specifically what to fix. Otherwise decide 'proceed'."""
        )
        self.just_guided = False

    async def steer_before_tool(self, *, agent, tool_use, **kwargs):
        # Tone is about what the customer reads, so never block a tool call here.
        return Proceed(reason="Tone guardrail only reviews final replies")

    async def steer_after_model(self, *, agent, message, stop_reason, **kwargs):
        reply = ""
        for block in message.get("content", []):
            if "text" in block:
                reply = reply + block["text"]
        if stop_reason != "end_turn" or not reply.strip():
            return Proceed(reason="Not a final reply")
        if self.just_guided:
            # Only one rewrite per reply, so the agent can't get stuck in a loop.
            self.just_guided = False
            return Proceed(reason="Already rewritten once")

        print("[TONE] 🔍 Evaluating agent response...")
        judge = Agent(system_prompt=self.system_prompt, model=self.model or agent.model, callback_handler=None)
        verdict = judge(
            "Grade this customer service reply:\n\n" + reply,
            structured_output_model=ToneVerdict,
        ).structured_output
        if verdict.decision == "guide":
            print("[TONE] ⚠️  Tone guided: " + verdict.reason)
            self.just_guided = True
            return Guide(reason="Rewrite your last reply to the customer. " + verdict.reason)
        print("[TONE] ✅ Tone OK")
        return Proceed(reason=verdict.reason)


tone_handler = ToneGuardrailHandler()
print("✅ ToneGuardrailHandler defined")

---

## Part 4: Full Agent with Skills + Steering

Combine everything: tools, skills, and both steering handlers.

In [ ]:
agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[
        AgentSkills(skills=["./skills"]),
        RefundWorkflowHandler(),
        tone_handler,
    ],
    system_prompt=SYSTEM_PROMPT,
)

# The steering handler will enforce: lookup → orders → refund
result = agent(
    "I'm customer C-1001. I want a refund for my wireless headphones order ORD-5521. "
    "Yes, please process it."
)

---

## 🎯 Try It Yourself

Test the steering enforcement — what happens if you try to skip steps?

In [ ]:
# Try asking for a refund without providing a customer ID
# The steering handler should force the agent to look up the customer first

agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[RefundWorkflowHandler(), tone_handler],
    system_prompt="You are a customer service agent. Process refunds immediately when asked.",
)

# Even with a permissive prompt, steering blocks the shortcut
result = agent("Process a refund for order ORD-5521, amount $79.99")

---

## 💬 Want a real multi-turn conversation?

In a notebook, each cell is a **single turn**. To chat back and forth with the agent — skills and steering handlers active on every turn — run the companion script in a **terminal**. From the cloned repo:

```bash
cd samples/03-skills-steering   # Workshop Studio: cd /workshop/samples/03-skills-steering
pip install -r requirements.txt
python chat.py
```

Type your messages, and `quit` (or Ctrl+C) to exit. The agent keeps its context across turns, so the refund workflow enforcement plays out over a real conversation.

---

## What's Next

The agent is smart and safe, but it forgets everything between sessions. In **Module 4: Session Managers**, you'll add persistence so the agent remembers previous conversations.